<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Outperfomance_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NIFTY STOCK OUTPERFORMANCE PROBABILITY
# AutoML + QuantStats | Leakage-Safe Version
# ============================================================

!pip -q install yfinance flaml quantstats

import yfinance as yf
import pandas as pd
import numpy as np
import quantstats as qs
from flaml import AutoML
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

# ------------------------------------------------------------
# 1. STOCK UNIVERSE
# ------------------------------------------------------------

stocks = [
"ADANIENT.NS","ADANIPORTS.NS","APOLLOHOSP.NS","ASIANPAINT.NS",
"AXISBANK.NS","BAJAJ-AUTO.NS","BAJFINANCE.NS","BAJAJFINSV.NS",
"BEL.NS","BHARTIARTL.NS","CIPLA.NS","COALINDIA.NS","DRREDDY.NS",
"EICHERMOT.NS","ETERNAL.NS","GRASIM.NS","HCLTECH.NS","HDFCBANK.NS",
"HDFCLIFE.NS","HEROMOTOCO.NS","HINDALCO.NS","HINDUNILVR.NS",
"ICICIBANK.NS","INDUSINDBK.NS","INFY.NS","ITC.NS","JIOFIN.NS",
"JSWSTEEL.NS","KOTAKBANK.NS","LT.NS","M&M.NS","MARUTI.NS",
"MAXHEALTH.NS","NESTLEIND.NS","NTPC.NS","ONGC.NS","POWERGRID.NS",
"RELIANCE.NS","SBILIFE.NS","SBIN.NS","SHRIRAMFIN.NS","SUNPHARMA.NS",
"TATACONSUM.NS","TATASTEEL.NS","TCS.NS","TECHM.NS",
"TITAN.NS","TRENT.NS","ULTRACEMCO.NS"
]

# Tata Motors is intentionally omitted because Yahoo's
# current ticker changed after its corporate restructuring.

# ------------------------------------------------------------
# 2. DOWNLOAD DATA
# ------------------------------------------------------------

prices = yf.download(
    stocks + ["^NSEI"],
    start="2015-01-01",
    auto_adjust=True,
    progress=False
)["Close"]

nifty = prices["^NSEI"].copy()
records = []

# ------------------------------------------------------------
# 3. FEATURE + TARGET CREATION
# ------------------------------------------------------------

for s in stocks:

    if s not in prices.columns:
        continue

    d = pd.DataFrame(index=prices.index)
    d["Close"] = prices[s]

    # Current information
    d["Ret_5"] = d["Close"].pct_change(5, fill_method=None)
    d["Ret_20"] = d["Close"].pct_change(20, fill_method=None)
    d["Momentum_60"] = d["Close"].pct_change(60, fill_method=None)

    d["Volatility"] = (
        d["Close"]
        .pct_change(fill_method=None)
        .rolling(20)
        .std()
    )

    d["Nifty_20D"] = nifty.pct_change(
        20, fill_method=None
    )

    # FUTURE performance — target only
    stock_future = (
        d["Close"].shift(-20) / d["Close"] - 1
    )

    nifty_future = (
        nifty.shift(-20) / nifty - 1
    )

    d["Target"] = (
        stock_future > nifty_future
    ).astype(int)

    d["Ticker"] = s

    records.append(d)

data = pd.concat(records).dropna()

features = [
    "Ret_5",
    "Ret_20",
    "Momentum_60",
    "Volatility",
    "Nifty_20D"
]

X = data[features]
y = data["Target"]

print("Dataset:", X.shape)
print("Outperformance Rate:",
      round(y.mean()*100,2), "%")

# ------------------------------------------------------------
# 4. TIME-BASED SPLIT
# ------------------------------------------------------------

dates = np.sort(data.index.unique())

split_date = dates[int(len(dates)*0.80)]

train = data[data.index < split_date]
test = data[data.index >= split_date]

X_train = train[features]
y_train = train["Target"]

X_test = test[features]
y_test = test["Target"]

# ------------------------------------------------------------
# 5. AUTOML
# ------------------------------------------------------------

automl = AutoML()

automl.fit(
    X_train,
    y_train,
    task="classification",
    metric="log_loss",
    time_budget=60,
    seed=42,
    eval_method="holdout"
)

print("\nBest AutoML Model:")
print(automl.model)

# ------------------------------------------------------------
# 6. PROBABILITY
# ------------------------------------------------------------

prob = automl.predict_proba(X_test)[:,1]

print("\nProbability Metrics")
print("-------------------")

print("ROC-AUC:",
      round(roc_auc_score(y_test,prob),4))

print("Log Loss:",
      round(log_loss(y_test,prob),4))

print("Brier Score:",
      round(brier_score_loss(y_test,prob),4))

# ------------------------------------------------------------
# 7. STOCK PROBABILITY RANKING
# ------------------------------------------------------------

test = test.copy()
test["Probability"] = prob

ranking = (
    test.groupby("Ticker")["Probability"]
    .last()
    .sort_values(ascending=False)
)

print("\nTOP STOCKS BY OUTPERFORMANCE PROBABILITY")
print("-----------------------------------------")

display(
    (ranking.head(15)*100)
    .round(2)
    .to_frame("Probability %")
)

# ------------------------------------------------------------
# 8. PROBABILITY BUCKET ANALYSIS
# ------------------------------------------------------------

test["Probability_Bucket"] = pd.cut(
    test["Probability"],
    bins=[0,0.4,0.5,0.6,0.7,1],
    labels=[
        "<40%",
        "40-50%",
        "50-60%",
        "60-70%",
        ">70%"
    ]
)

bucket = (
    test.groupby(
        "Probability_Bucket",
        observed=True
    )["Target"]
    .agg(["count","mean"])
)

bucket["Observed_Rate"] = bucket["mean"] * 100

print("\nPROBABILITY CALIBRATION")
display(bucket)

# ------------------------------------------------------------
# 9. CURRENT PORTFOLIO
# ------------------------------------------------------------

selected = ranking[ranking >= 0.60]

print("\nStocks with Probability >= 60%")
print(list(selected.index))

# ------------------------------------------------------------
# 10. NEXT-PERIOD PORTFOLIO TEST
# ------------------------------------------------------------

# Use signals at each date and calculate the NEXT
# 1-day stock return — no future 20D leakage.

test["Next_Return"] = (
    test.groupby("Ticker")["Close"]
    .shift(-1) /
    test["Close"] - 1
)

daily_portfolio = []

for date, group in test.groupby(level=0):

    picks = group[
        group["Probability"] >= 0.60
    ].copy()

    if len(picks) > 0:

        daily_portfolio.append({
            "Date": date,
            "Return": picks["Next_Return"].mean()
        })

portfolio = pd.DataFrame(
    daily_portfolio
).set_index("Date")

portfolio = portfolio["Return"].dropna()

# ------------------------------------------------------------
# 11. QUANTSTATS
# ------------------------------------------------------------

if len(portfolio) > 30:

    print("\nQuantStats — Probability Portfolio")
    print("-----------------------------------")

    print(
        "Sharpe:",
        round(qs.stats.sharpe(portfolio),3)
    )

    print(
        "CAGR:",
        round(qs.stats.cagr(portfolio)*100,2),
        "%"
    )

    print(
        "Max Drawdown:",
        round(
            qs.stats.max_drawdown(portfolio)*100,
            2
        ),
        "%"
    )

    print(
        "Volatility:",
        round(
            qs.stats.volatility(portfolio)*100,
            2
        ),
        "%"
    )

    qs.reports.basic(portfolio)

# ------------------------------------------------------------
# 12. FINAL DASHBOARD
# ------------------------------------------------------------

print("\n" + "="*55)
print("PROBABILISTIC STOCK SELECTION DASHBOARD")
print("="*55)

print(
    "Stocks ≥60% probability:",
    len(selected)
)

print(
    "Highest probability stock:",
    ranking.index[0]
)

print(
    "Highest probability:",
    round(ranking.iloc[0]*100,2),
    "%"
)

print("\nProject 8 completed.")

Dataset: (131194, 5)
Outperformance Rate: 51.48 %
[flaml.automl.logger: 09-16 02:42:41] {2375} INFO - task = classification
[flaml.automl.logger: 09-16 02:42:41] {2386} INFO - Evaluation method: holdout
[flaml.automl.logger: 09-16 02:42:41] {2489} INFO - Minimizing error metric: log_loss
[flaml.automl.logger: 09-16 02:42:41] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 09-16 02:42:41] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 09-16 02:42:41] {3046} INFO - Estimated sufficient time budget=2637s. Estimated necessary time budget=61s.
[flaml.automl.logger: 09-16 02:42:41] {3097} INFO -  at 0.2s,	estimator lgbm's best error=6.9165e-01,	best estimator lgbm's best error=6.9165e-01
[flaml.automl.logger: 09-16 02:42:41] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 09-16 02:42:41] {3097} INFO -  at 0.2s,	estimator lgbm's best error=6.9165e-01,	be

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 09-16 02:42:42] {3097} INFO -  at 0.3s,	estimator sgd's best error=6.9214e-01,	best estimator lgbm's best error=6.9165e-01
[flaml.automl.logger: 09-16 02:42:42] {2911} INFO - iteration 3, current learner sgd
[flaml.automl.logger: 09-16 02:42:42] {3097} INFO -  at 0.3s,	estimator sgd's best error=6.9214e-01,	best estimator lgbm's best error=6.9165e-01
[flaml.automl.logger: 09-16 02:42:42] {2911} INFO - iteration 4, current learner lgbm
[flaml.automl.logger: 09-16 02:42:42] {3097} INFO -  at 0.3s,	estimator lgbm's best error=6.9154e-01,	best estimator lgbm's best error=6.9154e-01
[flaml.automl.logger: 09-16 02:42:42] {2911} INFO - iteration 5, current learner lgbm
[flaml.automl.logger: 09-16 02:42:42] {3097} INFO -  at 0.4s,	estimator lgbm's best error=6.9054e-01,	best estimator lgbm's best error=6.9054e-01
[flaml.automl.logger: 09-16 02:42:42] {2911} INFO - iteration 6, current learner lgbm
[flaml.automl.logger: 09-16 02:42:42] {3097} INFO -  at 0.4s,	estimator lgb

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 09-16 02:43:00] {3097} INFO -  at 19.0s,	estimator lrl1's best error=6.9170e-01,	best estimator xgb_limitdepth's best error=6.7732e-01
[flaml.automl.logger: 09-16 02:43:00] {2911} INFO - iteration 51, current learner lrl1
[flaml.automl.logger: 09-16 02:43:00] {3097} INFO -  at 19.1s,	estimator lrl1's best error=6.9170e-01,	best estimator xgb_limitdepth's best error=6.7732e-01
[flaml.automl.logger: 09-16 02:43:00] {2911} INFO - iteration 52, current learner lrl1
[flaml.automl.logger: 09-16 02:43:00] {3097} INFO -  at 19.2s,	estimator lrl1's best error=6.9170e-01,	best estimator xgb_limitdepth's best error=6.7732e-01
[flaml.automl.logger: 09-16 02:43:00] {2911} INFO - iteration 53, current learner lrl1
[flaml.automl.logger: 09-16 02:43:01] {3097} INFO -  at 20.1s,	estimator lrl1's best error=6.9165e-01,	best estimator xgb_limitdepth's best error=6.7732e-01
[flaml.automl.logger: 09-16 02:43:01] {2911} INFO - iteration 54, current learner extra_tree
[flaml.automl.logg